# Day 1 — Seeing the Climate Through Data

**Welcome!** Today we explore real datasets to see how Earth's climate is changing — and why.

---

## Contents

- **PART I** — From raw numbers to global and local patterns
- **PART II** — Temperature on Earth is changing. What is driving these changes?
- **PART III** — Climate in Negros and the Philippines

## Setup

Run this cell first. It installs the workshop tools and connects to your Google Drive.

In [ ]:
# 1
import os                                                                          # in English:
if not os.path.exists("/content/USLS-Workshop"):                                   # IF workshop isn't downloaded yet:
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop  # download the workshop
%cd /content/USLS-Workshop                                                         # open the workshop
!pip install -q .                                                                  # install what is needed to do the workshop

from google.colab import drive
drive.mount('/content/drive')                                                      # connect to your Google Drive

In [ ]:
# 2
import numpy as np                                                      # anything with numbers
import pandas as pd                                                     # anything with tables
import xarray as xr                                                     # geospatial data
import matplotlib.pyplot as plt                                         # make visuals
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from workshop_utils import RAW_DIR, PROCESSED_DIR                       # RAW_DIR is data/raw/, PROCESSED_DIR is data/processed/

print("Ready!")

---

# PART I — From raw numbers to global and local patterns

## 1) What is 'climate data'?

Before we analyse anything, we need to understand what we are working with. In this workshop we use **real observational datasets** collected, cleaned, and published by scientific research groups.

Today we start with two widely used global temperature records:

1. **Berkeley Earth Surface Temperature (BEST)**
   A globally gridded land–ocean temperature record compiled from thousands of weather stations and ocean buoys.
   👉 https://berkeleyearth.org/data/

2. **NASA GISTEMP**
   A global temperature-anomaly record maintained by NASA's Goddard Institute for Space Studies.
   👉 https://data.giss.nasa.gov/gistemp/

Both records express temperature as an **[anomaly](https://en.wikipedia.org/wiki/Temperature_anomaly)**: the difference from the average temperature over a reference period (1951–1980 for both datasets). A positive anomaly means warmer than that baseline; negative means cooler. Using anomalies rather than absolute temperatures makes it easier to compare stations at different altitudes and latitudes.

We have already downloaded and lightly processed these datasets. Let's load and inspect them.

In [ ]:
# 3
# --- Load Berkeley Earth ---
best = pd.read_csv(PROCESSED_DIR / "best_yearly.csv")   # pd = pandas -> good for loading tabular data (= data with rows & columns). pd.read_csv() to read .csv files :)
best.head(10)                                           # show the first 10 columns

> 💬 **Discussion:** What do you think these numbers represent? What does `Year = 1850`, `Anomaly = -0.44` mean physically?

In [ ]:
# 4
# --- Load NASA GISTEMP ---
gistemp = pd.read_csv(PROCESSED_DIR / "gistemp_yearly.csv")
gistemp.head(10)

---

## 2) The global temperature trend — *the hockey-stick curve*

Now let's actually *see* the data. We'll use **matplotlib** — the most common Python library for making plots. The cell below shows a complete example. Read through it and try to understand each line before moving on.

The term *hockey stick* refers to the shape of the global temperature record: a long, nearly flat handle stretching back centuries, followed by a sharp upward blade from around 1980 onward.
👉 https://en.wikipedia.org/wiki/Hockey_stick_graph_(global_temperature)

In [ ]:
# 5
# =============================================================
# 💡 EXAMPLE — A complete matplotlib figure
# =============================================================

fig, ax = plt.subplots(figsize=(10, 4))   # create a blank figure and axes

ax.plot(best["Year"], best["Anomaly"],    # x-axis: years, y-axis: temperature anomaly
        color="tomato", linewidth=1.2,
        label="Berkeley Earth (BEST)")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")   # horizontal reference line at temperature anomaly 0°

ax.set_title("Global Mean Surface Temperature Anomaly")      # title of the plot
ax.set_xlabel("Year")                                        # label on the x-axis
ax.set_ylabel("Temperature anomaly (°C)")                    # label on the y-axis
ax.legend()                                                  # show the legend

plt.tight_layout()
plt.show()

> 💬 **Discussion:** What patterns do you notice? When does the warming really start accelerating?

---

## 3) Comparing two datasets

BEST and GISTEMP are produced by different scientific teams using different methods. Do they agree?

The example below shows how to add a **second line** and a **legend** to a plot.

In [ ]:
# 6
# =============================================================
# 💡 EXAMPLE — Adding a second line and legend
# =============================================================
# Suppose we have two data series, A and B:

years_example = [2000, 2001, 2002, 2003, 2004]
series_A = [0.41, 0.54, 0.63, 0.62, 0.54]
series_B = [0.44, 0.56, 0.61, 0.60, 0.56]

fig, ax = plt.subplots(figsize=(6, 3))

ax.plot(years_example, series_A, color="tomato",  label="Dataset A")
ax.plot(years_example, series_B, color="steelblue", label="Dataset B", linestyle="--")

ax.set_title("Two datasets compared")
ax.set_xlabel("Year")
ax.set_ylabel("Value")
ax.legend()   # <-- shows the labels we gave to each plot() call

plt.tight_layout()
plt.show()

### 🖊️ Your turn: overlay GISTEMP on the BEST record

Copy the structure from the example above. You already have:
- `best["Year"]` and `best["Anomaly"]` for the BEST data
- `gistemp["Year"]` and `gistemp["Anomaly"]` for GISTEMP

**Tasks:**
1. Plot BEST in red (`color="tomato"`) and GISTEMP in blue (`color="steelblue"`)
2. Add a title, x-label, y-label, and legend
3. Add the `axhline` at zero (copy it from the example above!)

In [ ]:
# 7
fig, ax = plt.subplots(figsize=(10, 4))

# Plot BEST — already done for you:
ax.plot(best["Year"], best["Anomaly"], color="tomato", linewidth=1.2, label="Berkeley Earth (BEST)")

# ✏️ Add the GISTEMP line below:
# ax.plot( ... )

# ✏️ Add an axhline at y=0:
# ax.axhline( ... )

# ✏️ Add title, axis labels, and legend:
ax.set_title("...")
ax.set_xlabel("...")
ax.set_ylabel("...")
ax.legend()

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Do the two datasets agree with each other? Would you trust them more or less because of this?

---

## 4) Smoothing noisy data

Year-to-year temperature varies a lot due to natural events like El Niño. A **moving average** smooths out short-term noise to reveal the underlying trend.

Remember that we loaded the BEST data as `best = pd.read_csv(PROCESSED_DIR / "best_yearly.csv")`? Pandas (pd) has an in-built function for this: `rolling()`. The example below shows a 10-year moving average — each point becomes the average of the surrounding 10 years.

In [ ]:
# 8
# 10-year moving average (rolling mean)
best_smooth = best["Anomaly"].rolling(window=10, center=True).mean()            # rolling() rolls over the data, here creating groups of 10. mean() then takes the average in each group.

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(best["Year"], best["Anomaly"],
        color="tomato", alpha=0.3, linewidth=1, label="BEST (raw)")
ax.plot(best["Year"], best_smooth,
        color="darkred", linewidth=2.5, label="BEST (10-yr average)")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_title("Global Temperature — Raw vs Smoothed")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (°C)")
ax.legend()

plt.tight_layout()
plt.show()

**Try it for yourself**: play around with different values for `window_size` below. What do you notice?

In [ ]:
# 9
# Play around with the window size a couple of times, what do you see?
window_size = ...
best_very_smooth = best["Anomaly"].rolling(window=window_size, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(best["Year"], best_very_smooth,
        color="darkred", linewidth=2.5, label="BEST (10-yr average)")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_title("Global Temperature — Raw vs Smoothed")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (°C)")
ax.legend()

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Why does smoothing help us understand the trend? What information do we lose when we smooth?

---

# PART II — Temperature is rising. What is driving these changes?

We see a clear warming trend. But *why* is it happening? Let's examine the most common proposed explanations one by one.

---

## 1) Has the sun become dramatically brighter?

The sun's energy output is called **Total Solar Irradiance (TSI)**. It follows an **11-year cycle** driven by **sunspot activity**.

**Sunspots** are temporary dark patches on the sun's surface caused by strong local magnetic fields that suppress convection and make that region slightly cooler than its surroundings. Despite being "dark", a sun with more sunspots is actually *slightly brighter overall*, because the sunspots are accompanied by even brighter surrounding regions called faculae. The number of sunspots rises and falls on a roughly 11-year cycle, producing a corresponding small oscillation in TSI.
👉 https://en.wikipedia.org/wiki/Sunspot

In the distant past, **grand solar minima** — prolonged periods of unusually low sunspot activity — contributed to cooling events like the [*Little Ice Age*](https://en.wikipedia.org/wiki/Little_Ice_Age) (roughly 1300–1850 CE).
👉 https://en.wikipedia.org/wiki/Solar_minimum

Let's look at the data from NOAA's satellite record.

In [ ]:
# 10
# --- Load Total Solar Irradiance (NOAA) ---
tsi         = pd.read_csv(PROCESSED_DIR / "tsi_monthly.csv", parse_dates=["time"])
tsi_yearly  = pd.read_csv(PROCESSED_DIR / "tsi_yearly.csv")

print(f"TSI record: {tsi_yearly['Year'].min()} – {tsi_yearly['Year'].max()}")

In [ ]:
# 11
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=False)

# Top panel: monthly TSI showing the 11-year cycle
axes[0].plot(tsi["time"], tsi["TSI"], color="goldenrod", linewidth=0.8)
axes[0].set_title("Total Solar Irradiance — monthly (shows 11-year sunspot cycle)")
axes[0].set_ylabel("TSI (W m⁻²)")

# Bottom panel: yearly TSI alongside the BEST temperature anomaly
ax_tsi = axes[1]
ax_temp = ax_tsi.twinx()   # second y-axis on the right

ax_tsi.plot(tsi_yearly["Year"], tsi_yearly["TSI"],
            color="goldenrod", linewidth=1.5,
            linestyle='--', label="TSI (left axis)")
ax_temp.plot(best["Year"], best["Anomaly"],
             color="tomato", linewidth=2, alpha=0.7, label="BEST temp (right axis)")

ax_tsi.set_title("Yearly TSI vs Global Temperature")
ax_tsi.set_xlabel("Year")
ax_tsi.set_ylabel("TSI (W m⁻²)", color="goldenrod")
ax_temp.set_ylabel("Temperature anomaly (°C)", color="tomato")

lines1, labels1 = ax_tsi.get_legend_handles_labels()
lines2, labels2 = ax_temp.get_legend_handles_labels()
ax_tsi.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Does the long-term trend in solar output match the warming trend?

---

## 2) Have volcanoes changed Earth's temperature?

Large volcanic eruptions can inject millions of tonnes of sulphur dioxide (SO₂) into the stratosphere, where it reacts with water vapour to form reflective **sulphate aerosols**. These particles scatter incoming sunlight and can **temporarily cool** the planet by 0.1–0.5°C, typically for 1–3 years.
👉 https://en.wikipedia.org/wiki/Volcanic_winter

The most dramatic 20th-century examples:
- **1963** — [Mount Agung](https://en.wikipedia.org/wiki/Mount_Agung) (Bali, Indonesia) — caused roughly 0.3°C of global cooling
- **1982** — [El Chichón](https://en.wikipedia.org/wiki/El_Chich%C3%B3n) (Chiapas, Mexico)
- **1991** — [Mount Pinatubo](https://en.wikipedia.org/wiki/Mount_Pinatubo) (Luzon, Philippines) — the largest eruption of the 20th century; cooled global temperatures by ~0.5°C for two years

We use the **Sato–Lacis Aerosol Optical Depth (AOD)** dataset from NASA GISS — a measure of how opaque the stratosphere is to incoming solar radiation. Higher values mean more aerosols and stronger cooling.
👉 https://en.wikipedia.org/wiki/Aerosol_optical_depth

In [ ]:
# 12
# --- Load stratospheric AOD (Sato-Lacis, NASA GISS) ---
aod_df     = pd.read_csv(PROCESSED_DIR / "aod_global.csv")
aod_years  = aod_df["year"].values
aod_global = aod_df["aod"]

print(f"AOD record: {int(aod_years[0])} – {int(aod_years[-1])}")

In [ ]:
# 13
fig, ax = plt.subplots(figsize=(11, 3.5))

ax.fill_between(aod_years, aod_global.values, color="slategray", alpha=0.7)
ax.plot(aod_years, aod_global.values, color="slategray", linewidth=0.8)

# Annotate the three major eruptions
for year, name in [(1963, "Agung"), (1982, "El Chichón"), (1991, "Pinatubo")]:
    ax.axvline(year, color="firebrick", linewidth=1, linestyle="--", alpha=0.7)
    ax.text(year + 0.3, aod_global.values.max() * 0.75, name,
            color="firebrick", fontsize=9, rotation=90, va="bottom")

ax.set_title("Global Stratospheric Aerosol Optical Depth — a proxy for volcanic activity")
ax.set_xlabel("Year")
ax.set_ylabel("AOD (stratospheric)")
ax.set_xlim(1850, 2020)

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Do you see sudden spikes after major eruptions? Could volcanoes explain the *long-term warming trend*?

---

## 3) Is it ENSO's fault?

The **El Niño–Southern Oscillation (ENSO)** is the most powerful year-to-year climate driver on Earth. It arises from coupled interactions between ocean surface temperatures and atmospheric circulation over the tropical Pacific.
👉 https://en.wikipedia.org/wiki/El_Ni%C3%B1o%E2%80%93Southern_Oscillation

During **El Niño** phases, a pool of unusually warm water spreads across the central and eastern Pacific, releasing large amounts of heat into the atmosphere and pushing global average temperatures up. During **La Niña**, cooler-than-normal water dominates the same region and global temperatures dip slightly. ENSO also shifts rainfall patterns and typhoon tracks across Southeast Asia.

The strength of ENSO is measured by the **Niño 3.4 index** — the average sea-surface temperature anomaly over a defined box in the central-eastern equatorial Pacific (5°N–5°S, 170°W–120°W). By convention, El Niño conditions are declared when this index exceeds +0.5°C for at least five consecutive months; La Niña when it falls below −0.5°C.

Below is a complete example showing how to plot the ENSO index. Then you'll create a combined figure.

In [ ]:
# 14
# --- Load ENSO (Niño 3.4 index, KNMI Climate Explorer) ---
nino_yearly = pd.read_csv(PROCESSED_DIR / "enso_yearly.csv")

print(f"ENSO record: {nino_yearly['Year'].min()} – {nino_yearly['Year'].max()}")

In [ ]:
# 15
# =============================================================
# 💡 EXAMPLE — Colouring above/below zero using fill_between
# =============================================================

fig, ax = plt.subplots(figsize=(11, 3))

years = nino_yearly["Year"]
enso  = nino_yearly["enso"]

# Shade positive (El Niño) years red, negative (La Niña) years blue
ax.fill_between(years, enso, 0,
                where=(enso > 0), color="tomato",    alpha=0.6, label="El Niño")
ax.fill_between(years, enso, 0,
                where=(enso < 0), color="steelblue", alpha=0.6, label="La Niña")
ax.plot(years, enso, color="black", linewidth=0.6)
ax.axhline(0, color="black", linewidth=0.8)

ax.set_title("ENSO Index (Niño 3.4) — El Niño and La Niña years")
ax.set_xlabel("Year")
ax.set_ylabel("Niño 3.4 anomaly (°C)")
ax.legend()

plt.tight_layout()
plt.show()

### 🖊️ Your turn: ENSO vs global temperature

Make a two-panel figure:
- **Top panel:** The ENSO index (copy + adjust from the example above)
- **Bottom panel:** The BEST global temperature anomaly

Use `sharex=True` so the x-axes are linked, and limit both panels to the same time period using `ax.set_xlim(1880, 2024)`. This lets you visually compare whether warm (El Niño) years match warm temperature anomalies.

In [ ]:
# 16
# ✏️ Instantiate a subplot with 2 rows and 1 column
#fig, axes = plt.subplots(..., ..., figsize=(11, 6), sharex=True)

# --- Top panel: ENSO index ---
ax_enso = axes[0]
# ✏️ Copy and adapt the fill_between plot from the example:
# ax_enso.fill_between( ... )
# ax_enso.fill_between( ... )
# ax_enso.axhline( ... )
ax_enso.set_ylabel("Niño 3.4 (°C)")
ax_enso.set_title("ENSO Index vs Global Temperature")

# --- Bottom panel: BEST temperature ---
ax_temp = axes[1]
# ✏️ Plot the BEST temperature anomaly (see PART I for how):
# ax_temp.plot( ... )
ax_temp.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax_temp.set_ylabel("Temperature anomaly (°C)")
ax_temp.set_xlabel("Year")

# Zoom into the overlapping period
ax_temp.set_xlim(1880, 2024)

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Do El Niño years (red) tend to coincide with temperature peaks? Can ENSO alone explain the long-term upward trend?

---

## 4) What about CO₂?

**Carbon dioxide (CO₂)** is a **greenhouse gas**: it absorbs outgoing infrared radiation emitted by Earth's surface and re-emits it in all directions, trapping heat in the lower atmosphere. This is the **greenhouse effect** — a natural process that keeps Earth roughly 33°C warmer than it would otherwise be. Human emissions are amplifying it beyond the natural baseline.
👉 https://en.wikipedia.org/wiki/Greenhouse_gas
👉 https://en.wikipedia.org/wiki/Greenhouse_effect

Since the **[Industrial Revolution](https://en.wikipedia.org/wiki/Industrial_Revolution)** (~1750), humans have burned coal, oil, and natural gas at an ever-increasing rate, releasing CO₂ that was locked underground for millions of years.

The **Keeling Curve** is the longest continuous instrumental record of atmospheric CO₂, measured at the **[Mauna Loa Observatory](https://en.wikipedia.org/wiki/Mauna_Loa_Observatory)** in Hawaii since 1958 by chemist **[Charles David Keeling](https://en.wikipedia.org/wiki/Charles_David_Keeling)**.
👉 https://en.wikipedia.org/wiki/Keeling_Curve

In [ ]:
# 17
# --- Load Mauna Loa CO₂ (NOAA Global Monitoring Laboratory) ---
co2        = pd.read_csv(PROCESSED_DIR / "co2_monthly.csv")
co2_yearly = pd.read_csv(PROCESSED_DIR / "co2_yearly.csv")

co2.head()

In [ ]:
# 18
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

# --- Top panel: Keeling Curve ---
ax_co2 = axes[0]
ax_co2.plot(co2["decimal date"], co2["average"],
            color="forestgreen", linewidth=0.8)
ax_co2.set_title("Atmospheric CO₂ — the Keeling Curve (Mauna Loa, Hawaii)")
ax_co2.set_ylabel("CO₂ (ppm)")
ax_co2.set_xlabel("Year")

# Annotate the 350 ppm threshold (often cited as a planetary boundary)
ax_co2.axhline(350, color="orange", linewidth=1, linestyle="--")
ax_co2.text(1960, 352, "350 ppm (planetary boundary)", color="orange", fontsize=8)

# Annotate 400 ppm crossed for first time
ax_co2.axhline(400, color="firebrick", linewidth=1, linestyle="--")
ax_co2.text(1960, 402, "400 ppm (first exceeded ~2013)", color="firebrick", fontsize=8)

# --- Bottom panel: CO₂ and temperature on the same time axis ---
ax_co2b = axes[1]
ax_temp2 = ax_co2b.twinx()

ax_co2b.plot(co2_yearly["year"], co2_yearly["average"],
             color="forestgreen", linewidth=1.8, label="CO₂ (left)")
ax_temp2.plot(best["Year"], best["Anomaly"],
              color="tomato", linewidth=1.2, alpha=0.8, label="BEST temp (right)")

ax_co2b.set_title("CO₂ concentration vs global temperature")
ax_co2b.set_xlabel("Year")
ax_co2b.set_ylabel("CO₂ (ppm)", color="forestgreen")
ax_temp2.set_ylabel("Temperature anomaly (°C)", color="tomato")

lines1, labels1 = ax_co2b.get_legend_handles_labels()
lines2, labels2 = ax_temp2.get_legend_handles_labels()
ax_co2b.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Notice the small wiggles in the monthly CO₂ record. What causes the regular annual up-down oscillation? *(Hint: think about seasons and photosynthesis.)*

---

## 5) Correlation vs causation ⚠️

CO₂ and temperature track each other very closely. Let's make a scatter plot to visualise this correlation.

In [ ]:
# 19
# Merge CO₂ and BEST on year (only keep years present in both datasets)
merged = co2_yearly.merge(best, left_on="year", right_on="Year", how="inner")

fig, ax = plt.subplots(figsize=(6, 5))

scatter = ax.scatter(merged["average"], merged["Anomaly"],
                     c=merged["year"], cmap="viridis",
                     s=40, edgecolors="none", alpha=0.85)
plt.colorbar(scatter, ax=ax, label="Year")

ax.set_title("CO₂ vs Global Temperature Anomaly")
ax.set_xlabel("CO₂ concentration (ppm)")
ax.set_ylabel("Temperature anomaly (°C)")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** If CO₂ is low, temperature is low. If CO₂ is high, temperature is high. Can you conclude global temperature is caused by atmospheric CO₂ concentration?

The correlation between CO₂ and temperature is striking. But does correlation mean causation?

Consider this: the number of Nicolas Cage films released per year also correlates with drowning deaths in swimming pools. Also: popularity of the first name Brooklyn correlates with UFO sightings in Kentucky.

Check out some more spurious correlations 👉 https://www.tylervigen.com/spurious-correlations

So why *do* scientists confidently say CO₂ causes warming?

1. **Physical mechanism**: The greenhouse properties of CO₂ were first demonstrated in the 1850s. [**Eunice Newton Foote**](https://en.wikipedia.org/wiki/Eunice_Newton_Foote) showed in 1856 that CO₂-filled cylinders heat more strongly in sunlight than air-filled ones. [**John Tyndall**](https://en.wikipedia.org/wiki/John_Tyndall) then proved in 1859 that CO₂ absorbs infrared radiation in precise laboratory experiments. CO₂ *provably* traps heat — this is not a model prediction but a directly measured physical property.
2. **Directionality**: [Isotopic analysis](https://en.wikipedia.org/wiki/Isotope_analysis) of atmospheric CO₂ reveals the fingerprint of fossil carbon: the ratio of ¹³C to ¹²C has been declining steadily, exactly as expected when ancient organic carbon (depleted in ¹³C) is burned and added to the atmosphere.
3. **Ruling out alternatives**: We have just seen that solar output and volcanoes cannot explain the long-term warming trend, and ENSO produces only year-to-year variation around the trend.

> 💬 **Discussion:** Which of these three points do you find most convincing? Why?

---

## *"But CO₂ fluctuates naturally — it's not actually humans' fault!"*

Fair point. CO₂ has varied naturally throughout Earth's history, driven by ice ages, volcanic outgassing, and changes in ocean chemistry. So how do we know today's rise is unusual?

The answer lies in **ice cores** — cylinders of ice drilled from the great ice sheets of Antarctica and Greenland. As snow accumulates and compresses over thousands of years, tiny **air bubbles** become trapped inside, preserving samples of the ancient atmosphere at the time the ice formed. By analysing these bubbles, scientists can reconstruct atmospheric CO₂ concentrations going back **800,000 years** — long before any human industrial activity.
👉 https://en.wikipedia.org/wiki/Ice_core

The record below comes from **EPICA Dome C** (*European Project for Ice Coring in Antarctica*) — one of the deepest ice cores ever drilled, reaching 3,270 m into East Antarctica and spanning eight complete glacial–interglacial cycles.
👉 https://en.wikipedia.org/wiki/European_Project_for_Ice_Coring_in_Antarctica

The temperature proxy (`delta_t_c`) is inferred from the **deuterium isotope ratio** (δD) of the ice water itself: warmer periods leave a distinct isotopic fingerprint compared to cold periods, allowing past temperatures to be reconstructed from chemistry alone.

The figure below places 800,000 years of natural variability in context. Pay attention to the *scale* of natural CO₂ variation — and where today's Mauna Loa measurements sit by comparison.

In [ ]:
# 20
# --- Load the EPICA Dome C ice-core data ---
epica = pd.read_csv(PROCESSED_DIR / "epica.csv")

# --- Build a combined figure with EPICA and Mauna Loa ---
fig, ax_epica = plt.subplots(figsize=(11, 5))

ax_epica.plot(epica["age_ka"], epica["delta_t_c"], color="#333333", linewidth=0.8)
ax_epica.fill_between(epica["age_ka"], epica["delta_t_c"], 0,
                      where=(epica["delta_t_c"] > 0), color="tomato",    alpha=0.4)
ax_epica.fill_between(epica["age_ka"], epica["delta_t_c"], 0,
                      where=(epica["delta_t_c"] < 0), color="steelblue", alpha=0.4)
ax_epica.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax_epica.set_xlim(epica["age_ka"].max(), epica["age_ka"].min())
ax_epica.set_title("EPICA Dome C temperature proxy over 800,000 years")
ax_epica.set_ylabel("Temperature difference (°C)")
ax_epica.set_xlabel("Age model (ka before present)")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** What was temperature on Earth like the past 800 000 years? Is it hot now, or cold? When was the last ice age, you think? Can you imagine our ancestors, having to live in a world 10 degrees colder...

The figure above showed *temperature* over 800,000 years from the EPICA ice core. Now let's add *CO₂* — measured from the same kind of ice cores — to see how the two track each other. The CO₂ record comes from the **[Vostok ice core](https://en.wikipedia.org/wiki/Vostok_Station)** in East Antarctica. The trapped air bubbles are analysed directly for their CO₂ content.

### 🖊️ Your turn: temperature and CO₂ together

Make a 2-panel figure (`sharex=True`) that puts both records on the same timeline:
- **Top panel:** EPICA temperature proxy — copy the red/blue `fill_between` style from the EPICA example above
- **Bottom panel:** Vostok CO₂ record

Both datasets use *ka before present* on the x-axis. Don’t forget to reverse the x-axis so that time runs left to right (oldest on the left).

In [ ]:
# 21
# --- Load Vostok CO₂ ice core record ---
vostok = pd.read_csv(PROCESSED_DIR / "vostok_co2.csv")
print(f"Vostok CO₂: {vostok['age_ka'].max():.0f} – {vostok['age_ka'].min():.0f} ka before present")

In [ ]:
# 22
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# --- Top panel: EPICA temperature ---
ax_t = axes[0]
# ✏️ Fill warm periods red and cold periods blue — look at cell 20 for the pattern:
# ax_t.fill_between( ... )
# ax_t.fill_between( ... )
# ax_t.plot( ... )
ax_t.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax_t.set_ylabel("Temperature anomaly (°C)")
ax_t.set_title("800,000 years of climate: temperature and CO₂")

# --- Bottom panel: Vostok CO₂ ---
ax_c = axes[1]
# ✏️ Plot the Vostok CO₂ record as a green line:
# ax_c.plot( ... )
ax_c.axhline(280, color="gray", linewidth=0.8, linestyle=":", label="280 ppm — typical interglacial peak")
ax_c.set_ylabel("CO₂ (ppm)")
ax_c.set_xlabel("Age (ka before present)")
ax_c.legend()

# ✏️ Reverse the x-axis so the oldest age is on the left:
# ax_c.set_xlim(???, ???)

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Do you see a pattern between temperature and CO₂? How have humans influenced atmospheric CO₂ concentrations since the industrial revolution? Were there any such concentrations in the past hundreds of thousands of years?

Throughout all eight glacial cycles in the record, CO₂ oscillated between roughly **180 ppm** (glacial, cold) and **280 ppm** (interglacial, warm) — never exceeding 300 ppm naturally. Currently, we sit at about **425-430 ppm**...

---

## *"So what? It's just a couple of degrees, and I don't care about polar bears either way. They're mean and eat cute seals."*

For the Philippines, even small shifts in global average temperature translate into:

- 🌊 **Sea level rise** — [thermal expansion](https://en.wikipedia.org/wiki/Thermal_expansion) of the ocean plus glacier and ice-sheet melt threatens low-lying coastal areas
- 🌀 **Stronger typhoons** — warmer oceans provide more energy; [Typhoon Haiyan (Yolanda, 2013)](https://en.wikipedia.org/wiki/Typhoon_Haiyan) made landfall with winds over 300 km/h, one of the strongest tropical cyclones ever recorded
- 🌧️ **More extreme rainfall** — warmer air holds more water vapour, intensifying floods and storms
- 🌾 **Heat stress on crops** — rice yield drops sharply above 35°C

In **Part III** we will zoom into the Philippines and Negros to make this concrete.

---

# PART III — Climate in Negros and the Philippines

## 1) What is 'climate'?

According to Wikipedia: *"Climate is the long-term weather pattern in a region, typically averaged over 30 years."*

The standard reference period used today is **1991–2020**. Let's look at the average temperature and precipitation across the Philippines during that period using **ERA5** — a global atmospheric **[reanalysis](https://en.wikipedia.org/wiki/Atmospheric_reanalysis)** dataset produced by the [European Centre for Medium-Range Weather Forecasts (ECMWF)](https://en.wikipedia.org/wiki/European_Centre_for_Medium-Range_Weather_Forecasts).

A reanalysis combines historical observations (weather stations, ships, radiosondes, satellites) with a numerical weather model to produce a consistent, gap-free, gridded record of the past atmosphere — filling in areas where direct measurements are sparse or absent.

In [ ]:
# 23
# --- Load ERA5 Philippines climatology (1991–2020) ---
t2m_mean       = xr.open_dataarray(PROCESSED_DIR / "t2m_clim_mean.nc")
tp_mean_annual = xr.open_dataarray(PROCESSED_DIR / "tp_clim_annual_mean.nc")
print("Done!")

In [ ]:
# 24
# =============================================================
# 💡 EXAMPLE — A 2D map of mean temperature over the Philippines
# =============================================================

lats = t2m_mean.latitude.values
lons = t2m_mean.longitude.values
lsm = xr.open_dataset(RAW_DIR / "ERA5" / "land_sea_mask" / "land_sea_mask_2024-01-01T00.nc")["land_sea_mask"].sel(latitude=lats, longitude=lons)

fig, ax = plt.subplots(figsize=(6, 8))

im = ax.pcolormesh(lons, lats, t2m_mean.values,
                   cmap="RdYlBu_r",                    # red = warm, blue = cool
                   vmin=20, vmax=30)                   # temperature range in °C

ax.contour(lsm.longitude, lsm.latitude, lsm.values,
           levels=[0.25, 0.5, 0.75],
           colors=["#666666", "#111111", "#666666"],
           linewidths=[0.6, 1.1, 0.6], alpha=0.7, zorder=4)

plt.colorbar(im, ax=ax, label="Mean temperature (°C)", shrink=0.7)

# Mark Bacolod
ax.scatter(122.95, 10.68, color="black", s=60, zorder=5)
ax.text(123.1, 10.68, "Bacolod", fontsize=9)

ax.set_title("Mean 2m Temperature — Philippines\n1991–2020 (ERA5)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Which regions are warmest? Which are coolest? Can you explain the pattern?

---

### 🖊️ Your turn: precipitation map

Using the temperature map above as a template, make the same kind of map for **mean annual precipitation** (`tp_mean_annual`). 

**Tips:**
- Use `cmap="YlGnBu"` (yellow = dry, blue = wet)
- The precipitation range is roughly 500–5000 mm/year; set `vmin=500, vmax=5000`
- Change the colorbar label to `"Mean annual precipitation (mm/yr)"`

In [ ]:
# 25
fig, ax = plt.subplots(figsize=(6, 8))

# ✏️ Create the pcolormesh for tp_mean_annual:
# im = ax.pcolormesh( ... )

# ✏️ Add the colorbar:
# plt.colorbar( ... )

# ✏️ Mark Bacolod:
# ax.scatter( ... )
# ax.text( ... )

ax.set_title("Mean Annual Precipitation — Philippines\n1991–2020 (ERA5)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Which regions are wettest? Driest? Why might the eastern and western sides of islands differ so strongly?

---

## 2) What climate does Bacolod have?

We'll extract temperature and precipitation data for **Bacolod** (10.68°N, 122.95°E) and look at:
1. The long-term temperature trend (1940–2024)
2. The typical annual cycle of temperature and rainfall (the *climatological* monthly means over 1991–2020)

In [ ]:
# 24
# --- Load Bacolod ERA5 time series ---
t2m_yearly  = xr.open_dataarray(PROCESSED_DIR / "t2m_bacolod_yearly.nc")
t2m_monthly = xr.open_dataarray(PROCESSED_DIR / "t2m_bacolod_monthly.nc")
tp_monthly  = xr.open_dataarray(PROCESSED_DIR / "tp_bacolod_monthly.nc")
print("Done!")

In [ ]:
# 27
# --- Long-term temperature trend in Bacolod ---
years = t2m_yearly["year"].values
temps = t2m_yearly.values

# Fit a linear trend
trend_coeffs = np.polyfit(years, temps, 1)
trend_line   = np.polyval(trend_coeffs, years)
warming_per_decade = trend_coeffs[0] * 10

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(years, temps, color="tomato", linewidth=1.2, alpha=0.8, label="Yearly mean T")
ax.plot(years, trend_line, color="darkred", linewidth=2.5, linestyle="--",
        label=f"Linear trend (+{warming_per_decade:.2f} °C / decade)")

ax.set_title("Annual Mean Temperature in Bacolod (ERA5, 1940–2024)")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature (°C)")
ax.legend()

plt.tight_layout()
plt.show()

> 💬 **Discussion:** How does the warming rate in Bacolod compare to the global average (~0.18 °C/decade since 1981)?

---

Now let's look at the **annual cycle** — the typical pattern of temperature and rainfall throughout the year. This is what characterises a place's *climate*.

In [ ]:
# 28
# =============================================================
# 💡 EXAMPLE — Monthly mean temperature (1991–2020 climatology)
# =============================================================

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(range(1, 13), t2m_monthly.values,
        color="tomato", linewidth=2, marker="o", markersize=6)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_ylim(24, 32)

ax.set_title("Monthly Mean Temperature — Bacolod\n1991–2020 climatology (ERA5)")
ax.set_xlabel("Month")
ax.set_ylabel("Temperature (°C)")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Is there a lot of tempearture variability throughout the year in Bacolod? Why do you think this is?

### 🖊️ Your turn: monthly precipitation

Make the same kind of plot for **monthly mean precipitation** (`tp_monthly`) — the average rainfall for each month of the year.

**Tips:**
- Use bars instead of a line: `ax.bar(range(1, 13), tp_monthly.values, color="steelblue")`
- Set y-axis label to `"Precipitation (mm/month)"`
- Use the same `month_names` list for x-tick labels

In [ ]:
# 29
fig, ax = plt.subplots(figsize=(8, 4))

# ✏️ Make a bar chart of monthly mean precipitation:
# ax.bar( ... )
ax.bar(range(1, 13), tp_monthly.values, color="steelblue", alpha=0.7)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)

ax.set_title("Monthly Mean Precipitation — Bacolod\n1991–2020 climatology (ERA5)")
ax.set_xlabel("Month")
ax.set_ylabel("Precipitation (mm/month)")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** Which months are wet and which are dry? What drives this seasonal pattern? *(Hint: think about the monsoon and the direction it comes from.)*

---

## 3) What is the Köppen-Geiger climate type of Bacolod?

The **[Köppen-Geiger (KG) classification](https://en.wikipedia.org/wiki/K%C3%B6ppen_climate_classification)** is the most widely used climate scheme in the world. It was developed by German-Russian climatologist [Wladimir Köppen](https://en.wikipedia.org/wiki/Wladimir_K%C3%B6ppen) in 1884 — as a botanist, he designed it around the idea that *climate shapes vegetation*. As a result, the KG classes correspond strongly to [biomes](https://en.wikipedia.org/wiki/Biome), making it especially useful in biology.

The system works as a **decision tree**. The five main climate groups are:

| Group | Name | Main criterion |
|-------|------|----------------|
| **A** | Tropical | Coldest month ≥ 18°C |
| **B** | Arid | Precipitation too low for forest |
| **C** | Temperate | Coldest month −3°C to 18°C |
| **D** | Continental | Coldest month < −3°C |
| **E** | Polar | Warmest month < 10°C |

Within **Group A**, the subtypes depend on rainfall seasonality:

| Subtype | Name | Criterion |
|---------|------|----------|
| **Af** | Tropical rainforest | Driest month ≥ 60 mm |
| **Am** | Tropical monsoon | Driest month < 60 mm, but annual P ≥ 25×(100 − driest month in mm) |
| **Aw** | Tropical savanna | Otherwise |

Let's compute the values we need from the ERA5 climatology.

In [ ]:
# 28
# Values needed for the classification
coldest_month_T  = float(t2m_monthly.min())          # minimum monthly temperature
annual_precip    = float(tp_monthly.sum() * 30.44)   # approx. annual total (mm)
driest_month_P   = float(tp_monthly.min() * 30.44)   # precipitation of driest month (mm)

print(f"Coldest month mean temperature : {coldest_month_T:.1f} °C")
print(f"Annual precipitation           : {annual_precip:.0f} mm")
print(f"Driest month precipitation     : {driest_month_P:.0f} mm")

Now let's write the decision tree in Python. The key tool is the `if / elif / else` structure.

Here's a simple example first:

In [ ]:
# 29
# =============================================================
# 💡 EXAMPLE — if / elif / else in Python
# =============================================================
temperature = 32

if temperature > 35:
    description = "Dangerously hot"
elif temperature > 30:
    description = "Very warm"
elif temperature > 20:
    description = "Comfortable"
else:
    description = "Cool"

print(f"At {temperature}°C: {description}")

# You can also combine conditions with 'and' / 'or':
driest = 35
annual = 1800

if driest < 60 and annual >= 25 * (100 - driest):
    print("This would be Am (tropical monsoon)")

### 🖊️ Your turn: classify Bacolod's climate

Using the three values computed above (`coldest_month_T`, `annual_precip`, `driest_month_P`) and the table of criteria, complete the decision tree below.

Work through the conditions in order:
1. First determine the main group (A, B, C, D, or E)
2. If it is Group A, determine the subtype (Af, Am, or Aw)

In [ ]:
# 30
# ✏️ Complete this decision tree:

# Step 1: determine main group
if coldest_month_T >= 18:
    group = "A"   # Tropical
# elif coldest_month_T >= ???:
#     group = "C"   # Temperate
# elif coldest_month_T >= ???:
#     group = "D"   # Continental
# else:
#     group = "E"   # Polar

# Step 2: if Group A, find the subtype
if group == "A":
    if driest_month_P >= 60:
        subtype = "f"   # Af — rainforest
    # elif driest_month_P < 60 and ???:
    #     subtype = "m"   # Am — monsoon
    # else:
    #     subtype = "w"   # Aw — savanna
else:
    subtype = ""   # (we won't classify further for non-A groups today)

climate_code = group + subtype
print(f"Bacolod's Köppen-Geiger climate type: {climate_code}")

> 💬 **Discussion:** According to Wikipedia, the Philippines has five climate types: tropical rainforest, tropical monsoon, tropical savanna, humid subtropical and oceanic. What would you expect for Bacolod? Does your result match?

---

## 4) Climate zones in the Philippines — present and future

To finish, let's zoom out. The Köppen-Geiger system has been applied globally, and researchers have used climate models to project how these zones will shift under continued warming.

[Beck et al. (2023)](https://doi.org/10.1038/s41597-023-02549-6) published 1 km resolution KG maps for 1901–2099 based on [CMIP6](https://en.wikipedia.org/wiki/Coupled_Model_Intercomparison_Project) climate model projections:

> Beck, H. E., McVicar, T. R., Vergopolan, N., Berg, A., Lutsko, N. J., Dufour, A., ... & Miralles, D. G. (2023). High-resolution (1 km) Köppen-Geiger maps for 1901–2099 based on constrained CMIP6 projections. *Scientific Data, 10*(1), 724.

Below is an example of how climate zones across the Philippines might shift by 2100 under a high-emissions scenario. Notice how tropical rainforest areas (Af) may contract, with savanna-type conditions (Aw) expanding as rainfall seasonality increases.

*(Interactive maps and downloads available at: https://www.gloh2o.org/koppen/)*

In [ ]:
# Standard Köppen-Geiger color scheme (Beck et al. 2023)
KG_INFO = [
    (1,  "Af",  "#0000FF"), (2,  "Am",  "#0078FF"), (3,  "Aw",  "#46AAFA"),
    (4,  "BWh", "#FF0000"), (5,  "BWk", "#FF9696"), (6,  "BSh", "#F5A500"),
    (7,  "BSk", "#FFD37F"), (8,  "Csa", "#FFFF00"), (9,  "Csb", "#C8C800"),
    (10, "Csc", "#969600"), (11, "Cwa", "#96FF00"), (12, "Cwb", "#64C800"),
    (13, "Cwc", "#329600"), (14, "Cfa", "#C8FF00"), (15, "Cfb", "#64FF00"),
    (16, "Cfc", "#32C800"), (17, "Dsa", "#FF00FF"), (18, "Dsb", "#C800C8"),
    (19, "Dsc", "#963296"), (20, "Dsd", "#966496"), (21, "Dwa", "#AB00AB"),
    (22, "Dwb", "#780078"), (23, "Dwc", "#4B0082"), (24, "Dwd", "#320050"),
    (25, "Dfa", "#00FFFF"), (26, "Dfb", "#37C8FF"), (27, "Dfc", "#007D7D"),
    (28, "Dfd", "#004B4B"), (29, "ET",  "#B2B2B2"), (30, "EF",  "#686868"),
]
cmap = mcolors.ListedColormap([c for _, _, c in KG_INFO])
norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, 31.5), ncolors=30)

# --- Present climate: 1991–2020 ---
ds_present = xr.open_dataset(PROCESSED_DIR / "kg_philippines_present.nc")
kg_present = ds_present[list(ds_present.data_vars)[0]].squeeze()

lat_name = "latitude" if "latitude" in kg_present.coords else "lat"
lon_name = "longitude" if "longitude" in kg_present.coords else "lon"
lats = kg_present[lat_name].values
lons = kg_present[lon_name].values

shown   = [int(v) for v in sorted(np.unique(kg_present.values)) if not np.isnan(v)]
patches = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in shown]

fig, ax = plt.subplots(figsize=(7, 9))
ax.pcolormesh(lons, lats, kg_present.values, cmap=cmap, norm=norm)
ax.set_title("Present climate (1991–2020)", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
fig.legend(handles=patches, loc="lower center", ncol=min(len(patches), 6),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Köppen-Geiger Climate Classification — Philippines\n(Beck et al. 2023)", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

In [ ]:
# --- Future climate: 2071–2099, SSP4-6.0 ---
ds_future  = xr.open_dataset(PROCESSED_DIR / "kg_philippines_future_ssp460.nc")
kg_future  = ds_future[list(ds_future.data_vars)[0]].squeeze()

shown_f   = [int(v) for v in sorted(np.unique(kg_future.values)) if not np.isnan(v)]
patches_f = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in shown_f]

fig, ax = plt.subplots(figsize=(7, 9))
ax.pcolormesh(lons, lats, kg_future.values, cmap=cmap, norm=norm)
ax.set_title("Future climate (2071–2099, SSP4-6.0)", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
fig.legend(handles=patches_f, loc="lower center", ncol=min(len(patches_f), 6),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Köppen-Geiger Climate Classification — Philippines\n(Beck et al. 2023)", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

> 💬 **Final discussion:**
> - What does a shift in Köppen-Geiger zone mean for the ecosystems and agriculture of a region?
> - If Bacolod's climate shifts from Am to Aw, what would change about day-to-day life and biodiversity here?
> - What can individuals, communities, and governments do in response?

---

## Well done!

Today you:
- Loaded and visualised real climate data from NASA, NOAA, and ECMWF
- Saw how different potential climate drivers (sun, volcanoes, ENSO, CO₂) compare to the observed warming trend
- Explored the climate of the Philippines and Bacolod using ERA5 reanalysis data
- Classified Bacolod according to the Köppen-Geiger system
- Got acquainted with Python, and concepts such as `variables`, `functions` and `classes`
- Learned to work with a bunch of core Python libraries: `numpy`, `pandas`, `xarray`, and `matplotlib`

**Tomorrow (Day 2)** we'll go further into biological data — species distributions, ecological surveys, and what data science can tell us about biodiversity.